In [1]:
# 最简单但有效的版本
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# 加载数据
train = pd.read_csv('/Users/caierchang/Downloads/train.csv')
test = pd.read_csv('/Users/caierchang/Downloads/test.csv')

# 只做最重要的处理
def simple_features(df):
    df = df.copy()

    # 1. 性别
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

    # 2. 填充年龄和票价
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())

    # 3. 登船港口
    df['Embarked'] = df['Embarked'].fillna('S')
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

    # 4. 家庭大小
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # 5. 是否儿童
    df['IsChild'] = (df['Age'] < 12).astype(int)

    # 6. 票价每人
    df['FarePerPerson'] = df['Fare'] / df['FamilySize']

    return df

# 处理数据
train_df = simple_features(train)
test_df = simple_features(test)

# 选择特征
features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'IsAlone', 'IsChild', 'FarePerPerson']

X_train = train_df[features]
y_train = train_df['Survived']
X_test = test_df[features]

# 简单随机森林
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

output = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

# 应用硬规则
output.loc[(test['Pclass'] == 1) & (test['Sex'] == 'female'), 'Survived'] = 1
output.loc[(test['Pclass'] == 3) & (test['Sex'] == 'male') & (test['Age'] > 15), 'Survived'] = 0

output.to_csv('/Users/caierchang/Desktop/final_submission.csv', index=False)